In [0]:
df = spark.read.table('medical_catalog.bronze.encounters')


In [0]:
# Convert columns to snakecase
for col in df.columns:
    new_col = col.lower().replace(' ', '_')
    if new_col != col:
        df = df.withColumnRenamed(col, new_col)


In [0]:
# type conversion
from pyspark.sql.functions import col
df = df.withColumn("base_encounter_cost", col("base_encounter_cost").cast("double"))
df = df.withColumn("total_claim_cost", col("total_claim_cost").cast("double"))
df = df.withColumn('Payer_coverage',col("payer_coverage").cast("double"))
# Convert to pandas and display
df.limit(10).toPandas()
print(df)




In [0]:
# Handling nulls
from pyspark.sql.functions import when

df = df.withColumn("reasoncode", when(col("reasoncode").isNull(),0.00 ).otherwise(col("reasoncode")))
df = df.withColumn("reasondescription", when(col("reasondescription").isNull(), "Unknown").otherwise(col("reasondescription")))
df.display(df.limit(10))

In [0]:
# Writing Data in Silver Table
df.write.mode("overwrite").saveAsTable("medical_catalog.silver.encounters")